# Toxic Comment Detection with Scikit-learn

This notebook implements a **multi-label toxic comment classifier** using the
Jigsaw Toxic Comment Classification Challenge dataset.

## Goal

The goal is to investigate how well a classical machine-learning pipeline based on
**TF-IDF features** and **Logistic Regression** can identify six forms of toxicity
in English online comments.

## Research questions

1. How effectively can TF-IDF features combined with Logistic Regression detect
   the six toxicity categories in the Jigsaw dataset?
2. How does performance differ between frequent and rare toxicity labels?
3. What kinds of comments cause errors, especially comments containing negation?

## Pipeline

1. Load and filter the dataset
2. Split data into training, validation, and test sets
3. Convert comments into TF-IDF features
4. Train six Logistic Regression classifiers using One-vs-Rest
5. Evaluate quantitatively
6. Perform qualitative error analysis
7. Save the trained model and vectorizer
8. Optionally launch a Gradio interface

The six labels are:

- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate


## 1. Imports

Run this cell first. If a package is missing, install it in the VS Code terminal with:

```bash
pip install pandas scikit-learn joblib gradio
```


In [1]:
import os
import joblib
import pandas as pd

from data_loader import load_jigsaw_data, LABELS

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    hamming_loss
)


## 2. Project settings

The project uses a **relative data path** so the repository can run on another
computer after the dataset is downloaded into the `data/` folder.

See `data/README.md` for the dataset setup instructions.


In [2]:
DATA_FOLDER = os.path.join(
    "data",
    "jigsaw-toxic-comment-classification-challenge"
)

THRESHOLD = 0.5


## 3. Load and filter the dataset

The helper module `data_loader.py` searches for `train.csv`, loads it, and
returns the comment text together with the six toxicity labels.


In [5]:
df, X, y, DATA_PATH = load_jigsaw_data(DATA_FOLDER)

print("Using:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()


Using:
data\jigsaw-toxic-comment-classification-challenge\train.csv\train.csv

Dataset shape:
(159571, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


## 4. Inspect labels

`X` contains the raw comments and `y` contains the six binary toxicity labels.

A comment may belong to more than one category at the same time, so this is a
**multi-label classification** problem.


In [6]:
print("Number of comments:", len(X))
print("Labels:", LABELS)

print("\nLabel frequencies:")
print(y.sum().sort_values(ascending=False))


Number of comments: 159571
Labels: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Label frequencies:
toxic            15294
obscene           8449
insult            7877
severe_toxic      1595
identity_hate     1405
threat             478
dtype: int64


## 5. Train / validation / test split

We use:

- **70% training**
- **20% validation**
- **10% test**

`random_state=42` makes the split reproducible.


In [7]:
# First reserve 10% for the final test set.
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42
)

# Split the remaining 90% into 70% training and 20% validation.
# 20 / 90 = 2 / 9
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=2 / 9,
    random_state=42
)

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Test examples:", len(X_test))

print("\nProportions:")
print("Train:", len(X_train) / len(X))
print("Validation:", len(X_val) / len(X))
print("Test:", len(X_test) / len(X))


Training examples: 111699
Validation examples: 31914
Test examples: 15958

Proportions:
Train: 0.6999956132379944
Validation: 0.19999874663942696
Test: 0.10000564012257866


## 6. TF-IDF vectorization

Scikit-learn classifiers cannot directly train on raw strings.

`TfidfVectorizer` converts the comments into numerical features.

We use both:

- **unigrams** such as `stupid`
- **bigrams** such as `not stupid` and `you are`

The vocabulary is learned **only from the training set**.  
Validation and test data are transformed using that same vocabulary.


In [8]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("Creating TF-IDF features...")

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("Finished.")

print("\nTraining matrix shape:", X_train_tfidf.shape)
print("Validation matrix shape:", X_val_tfidf.shape)
print("Test matrix shape:", X_test_tfidf.shape)


Creating TF-IDF features...
Finished.

Training matrix shape: (111699, 50000)
Validation matrix shape: (31914, 50000)
Test matrix shape: (15958, 50000)


## 7. Create the model

`LogisticRegression` is a binary classifier.

Because we have six labels, `OneVsRestClassifier` trains one Logistic Regression classifier for each label.

Conceptually:

- toxic → classifier 1
- severe_toxic → classifier 2
- obscene → classifier 3
- threat → classifier 4
- insult → classifier 5
- identity_hate → classifier 6


In [9]:
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=42
    ),
    n_jobs=1
)

model


,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it i

## 8. Train the model

Training is performed with one call to `fit()`.

`max_iter=1000` is the maximum number of optimization iterations allowed for
each Logistic Regression classifier.


In [10]:
print("Training model...")

model.fit(
    X_train_tfidf,
    y_train
)

print("Training finished.")


Training model...
Training finished.


## 9. Validation evaluation

The validation set is used to inspect model performance before using the final test set.

`classification_report` provides precision, recall, and F1-score for each toxicity label.


In [11]:
val_predictions = model.predict(X_val_tfidf)

print(
    classification_report(
        y_val,
        val_predictions,
        target_names=LABELS,
        zero_division=0
    )
)


               precision    recall  f1-score   support

        toxic       0.94      0.56      0.70      3140
 severe_toxic       0.57      0.24      0.34       315
      obscene       0.93      0.57      0.71      1698
       threat       0.79      0.11      0.19       104
       insult       0.83      0.49      0.61      1567
identity_hate       0.71      0.14      0.23       264

    micro avg       0.90      0.51      0.65      7088
    macro avg       0.79      0.35      0.46      7088
 weighted avg       0.89      0.51      0.64      7088
  samples avg       0.05      0.04      0.05      7088



## 10. Overall validation metrics

For a multi-label problem, it is useful to report more than one metric.

Here we calculate micro-averaged precision, recall, and F1.


In [12]:
val_precision = precision_score(
    y_val,
    val_predictions,
    average="micro",
    zero_division=0
)

val_recall = recall_score(
    y_val,
    val_predictions,
    average="micro",
    zero_division=0
)

val_f1 = f1_score(
    y_val,
    val_predictions,
    average="micro",
    zero_division=0
)

print(f"Validation Micro Precision: {val_precision:.4f}")
print(f"Validation Micro Recall:    {val_recall:.4f}")
print(f"Validation Micro F1:        {val_f1:.4f}")


Validation Micro Precision: 0.8965
Validation Micro Recall:    0.5106
Validation Micro F1:        0.6506


## 11. Final test evaluation

The test set should be used only after the model and preprocessing choices are fixed.


In [13]:
test_predictions = model.predict(X_test_tfidf)

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=LABELS,
        zero_division=0
    )
)


               precision    recall  f1-score   support

        toxic       0.93      0.58      0.71      1480
 severe_toxic       0.59      0.23      0.33       148
      obscene       0.94      0.56      0.71       836
       threat       1.00      0.11      0.20        37
       insult       0.85      0.48      0.62       791
identity_hate       0.81      0.09      0.16       147

    micro avg       0.90      0.51      0.65      3439
    macro avg       0.85      0.34      0.45      3439
 weighted avg       0.89      0.51      0.64      3439
  samples avg       0.05      0.04      0.04      3439



## 12. Additional multi-label metrics

- **Micro F1** weighs common labels more heavily.
- **Macro F1** gives equal importance to each label, including rare labels.
- **Subset accuracy** is very strict: all six labels for one comment must be correct.
- **Hamming loss** measures the fraction of individual label decisions that are incorrect.


In [14]:
micro_precision = precision_score(
    y_test,
    test_predictions,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_test,
    test_predictions,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_test,
    test_predictions,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0
)

subset_accuracy = accuracy_score(
    y_test,
    test_predictions
)

hamming = hamming_loss(
    y_test,
    test_predictions
)

print(f"Micro Precision: {micro_precision:.4f}")
print(f"Micro Recall:    {micro_recall:.4f}")
print(f"Micro F1:        {micro_f1:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Subset Accuracy: {subset_accuracy:.4f}")
print(f"Hamming Loss:    {hamming:.4f}")


Micro Precision: 0.9026
Micro Recall:    0.5118
Micro F1:        0.6532
Macro F1:        0.4530
Subset Accuracy: 0.9211
Hamming Loss:    0.0195


### Quantitative interpretation

The final test results show **high micro precision (0.9026)** but noticeably
lower **micro recall (0.5118)**. This means that predictions marked as toxic are
usually correct, but the classifier misses a substantial number of positive
toxicity labels.

Performance also differs strongly between labels. The common labels `toxic`,
`obscene`, and `insult` obtain much higher F1 scores than the rarer labels
`threat` and `identity_hate`. This pattern is consistent with the strong class
imbalance in the dataset.

Subset accuracy is reported for completeness, but it should not be interpreted
alone because most label values in this dataset are zero. Precision, recall,
F1, and per-label results are more informative for this task.


## 13. Prediction probabilities

`predict()` gives binary predictions.

`predict_proba()` gives the estimated probability for each toxicity category.


In [15]:
test_probabilities = model.predict_proba(X_test_tfidf)

print("Probability matrix shape:")
print(test_probabilities.shape)


Probability matrix shape:
(15958, 6)


## 14. Function for testing custom comments

The function below:

1. transforms a new comment with the trained TF-IDF vectorizer,
2. gets six toxicity probabilities,
3. compares each probability to the threshold,
4. returns a readable result.


In [16]:
def score_comment(comment, threshold=THRESHOLD):
    vectorized_comment = vectorizer.transform([comment])

    probabilities = model.predict_proba(vectorized_comment)[0]

    result = ""

    for label, probability in zip(LABELS, probabilities):
        prediction = probability >= threshold

        result += (
            f"{label}: "
            f"{probability:.4f} "
            f"-> {prediction}\n"
        )

    return result


## 15. Qualitative error analysis

The examples below are used to inspect how the classifier handles short toxic
expressions and **negation**.

Pay particular attention to:

- `not fat`
- `you are not fat`
- `not stupid`
- `you are not stupid`

Because TF-IDF is based on lexical and n-gram features, the model may strongly
associate words such as `stupid`, `fat`, or `dumb` with toxicity even when they
appear in a negated phrase.


In [17]:
comments = [
    "I love you",
    "you are stupid",
    "you are not stupid",
    "not stupid",
    "you are fat",
    "you are not fat",
    "not fat",
    "you are dumb",
    "you are not dumb"
]

for comment in comments:
    print("=" * 50)
    print("COMMENT:")
    print(comment)
    print()
    print(score_comment(comment))


COMMENT:
I love you

toxic: 0.0874 -> False
severe_toxic: 0.0128 -> False
obscene: 0.0451 -> False
threat: 0.0047 -> False
insult: 0.0441 -> False
identity_hate: 0.0150 -> False

COMMENT:
you are stupid

toxic: 0.9981 -> True
severe_toxic: 0.0456 -> False
obscene: 0.7446 -> True
threat: 0.0121 -> False
insult: 0.9830 -> True
identity_hate: 0.0683 -> False

COMMENT:
you are not stupid

toxic: 0.9996 -> True
severe_toxic: 0.0460 -> False
obscene: 0.9067 -> True
threat: 0.0116 -> False
insult: 0.9957 -> True
identity_hate: 0.0884 -> False

COMMENT:
not stupid

toxic: 0.9998 -> True
severe_toxic: 0.0237 -> False
obscene: 0.9139 -> True
threat: 0.0085 -> False
insult: 0.9917 -> True
identity_hate: 0.0531 -> False

COMMENT:
you are fat

toxic: 0.9859 -> True
severe_toxic: 0.1633 -> False
obscene: 0.7378 -> True
threat: 0.0122 -> False
insult: 0.9861 -> True
identity_hate: 0.1913 -> False

COMMENT:
you are not fat

toxic: 0.9325 -> True
severe_toxic: 0.0599 -> False
obscene: 0.4846 -> False
t

### Qualitative interpretation

The model correctly gives low toxicity probabilities to the clearly non-toxic
example `I love you` and high toxicity probabilities to direct insults such as
`you are stupid`.

However, negation remains a clear failure case. Phrases such as `you are not
stupid` and `not stupid` can still receive very high toxicity probabilities.
For `you are not fat`, negation lowers some probabilities, but the comment is
still predicted as toxic/insulting.

This suggests that unigram and bigram TF-IDF features capture useful lexical
patterns but do not provide a full semantic representation of negation.
This is an important limitation of the current system.


## 16. Save the trained model and vectorizer

Both objects are necessary later:

- the model performs the classification,
- the vectorizer converts new text into the same TF-IDF representation used during training.


In [17]:
os.makedirs("models", exist_ok=True)

MODEL_PATH = os.path.join(
    "models",
    "toxicity_sklearn_model.joblib"
)

VECTORIZER_PATH = os.path.join(
    "models",
    "tfidf_vectorizer.joblib"
)

joblib.dump(model, MODEL_PATH)
joblib.dump(vectorizer, VECTORIZER_PATH)

print("Saved model to:")
print(MODEL_PATH)

print("\nSaved vectorizer to:")
print(VECTORIZER_PATH)


Saved model to:
models\toxicity_sklearn_model.joblib

Saved vectorizer to:
models\tfidf_vectorizer.joblib


## 17. Load the saved model later

After the model has been trained and saved once, you can load it without retraining.


In [ ]:
# Run this in a fresh session if you want to use the saved files.

# loaded_model = joblib.load("models/toxicity_sklearn_model.joblib")
# loaded_vectorizer = joblib.load("models/tfidf_vectorizer.joblib")


## 18. Optional Gradio interface

Run this only after the model has been trained.

If Gradio is not installed:

```bash
pip install gradio
```


In [18]:
import gradio as gr

interface = gr.Interface(
    fn=score_comment,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Write a comment..."
    ),
    outputs=gr.Textbox(
        label="Toxicity prediction"
    ),
    title="Toxic Comment Detection",
    description=(
        "Multi-label toxic comment classification using "
        "TF-IDF and Logistic Regression."
    )
)

interface.launch()


c:\Users\yazar\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 19. Summary

This project demonstrates a complete classical NLP classification pipeline using
techniques from the course:

- scikit-learn
- train/validation/test splitting
- TF-IDF text representation
- One-vs-Rest classification
- Logistic Regression
- multi-label evaluation
- model persistence with joblib
- qualitative error analysis

The main strengths are high precision and a simple, reproducible pipeline.
The main limitations are class imbalance, low recall for rare labels, and weak
handling of semantic phenomena such as negation.
